In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, sum as _sum, avg, count, countDistinct, round, 
    dense_rank, row_number, lag, date_format
)

In [7]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Retail Analytics Pipeline") \
    .getOrCreate()

# Read CSV files from HDFS
path = "/user/student/retail/raw/"

customers = spark.read.csv(path + "customers.csv", header=True, inferSchema=True)
employees = spark.read.csv(path + "employees.csv", header=True, inferSchema=True)
orders = spark.read.csv(path + "orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv(path + "order_items.csv", header=True, inferSchema=True)
products = spark.read.csv(path + "products.csv", header=True, inferSchema=True)
stores = spark.read.csv(path + "stores.csv", header=True, inferSchema=True)

# Register Temp Views for Spark SQL
customers.createOrReplaceTempView("customers")
employees.createOrReplaceTempView("employees")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")
stores.createOrReplaceTempView("stores")

print("All DataFrames loaded and Temp Views created successfully!")

All DataFrames loaded and Temp Views created successfully!


In [9]:
# Q1: Revenue, profit, and profit margin (%) per product category
q1_df = order_items.join(products, "product_id") \
    .groupBy("category") \
    .agg(
        round(_sum(col("quantity") * col("unit_price")), 2).alias("total_revenue"),
        round(_sum(col("quantity") * (col("unit_price") - col("cost_price"))), 2).alias("total_profit")
    ) \
    .withColumn("profit_margin_pct", round((col("total_profit") / col("total_revenue")) * 100, 2))

q1_df.show(5)

+-----------------+-------------+------------+-----------------+
|         category|total_revenue|total_profit|profit_margin_pct|
+-----------------+-------------+------------+-----------------+
|             Food|       895.61|      388.01|            43.32|
|     Toys & Games|       2380.0|      994.19|            41.77|
|Sports & Outdoors|     22383.67|     9130.78|            40.79|
|      Electronics|     34708.69|    17532.07|            50.51|
|         Clothing|      5283.41|     2148.25|            40.66|
+-----------------+-------------+------------+-----------------+
only showing top 5 rows



In [14]:
# Q2: Top five customers in each state based on total spending
customer_spend = orders.join(order_items, "order_id") \
    .join(customers, "customer_id") \
    .groupBy("state", "customer_id", "first_name", "last_name") \
    .agg(_sum(col("quantity") * col("price_per_unit")).alias("total_spending"))

w_state = Window.partitionBy("state").orderBy(col("total_spending").desc())
q2_df = customer_spend.withColumn("rank", dense_rank().over(w_state)).filter(col("rank") <= 5)

q2_df.show(5)

+-----+-----------+----------+---------+-----------------+----+
|state|customer_id|first_name|last_name|   total_spending|rank|
+-----+-----------+----------+---------+-----------------+----+
| Ohio|         50|     Sarah|    Lewis|7952.360000000001|   1|
| Ohio|         41|   Anthony| Thompson|          1563.77|   2|
| Ohio|         39|  Jennifer|   Garcia|           626.53|   3|
|Texas|         24|    Sandra|   Garcia|          2928.61|   1|
|Texas|         36|   Dorothy|    Scott|          2169.93|   2|
+-----+-----------+----------+---------+-----------------+----+
only showing top 5 rows



In [15]:
# Q3: Store-level metrics (Revenue, Profit, Completed Orders, Unique Customers)
q3_df = orders.join(order_items, "order_id") \
    .join(products, "product_id") \
    .groupBy("store_id") \
    .agg(
        round(_sum(col("quantity") * col("price_per_unit")), 2).alias("total_revenue"),
        round(_sum(col("quantity") * (col("price_per_unit") - col("cost_price"))), 2).alias("total_profit"),
        countDistinct(col("order_id")).alias("completed_orders"),
        countDistinct(col("customer_id")).alias("unique_customers")
    )

q3_df.show(5)

+--------+-------------+------------+----------------+----------------+
|store_id|total_revenue|total_profit|completed_orders|unique_customers|
+--------+-------------+------------+----------------+----------------+
|      31|       392.36|      177.84|               1|               1|
|      85|      1216.81|      504.86|               1|               1|
|      78|       634.13|      212.62|               1|               1|
|      34|       852.82|      356.88|               1|               1|
|      28|        46.78|       14.95|               1|               1|
+--------+-------------+------------+----------------+----------------+
only showing top 5 rows



In [16]:
# Q4: Employees whose sales are higher than store average
emp_sales = orders.join(order_items, "order_id") \
    .groupBy("store_id", "employee_id") \
    .agg(_sum(col("quantity") * col("price_per_unit")).alias("emp_total_sales"))

w_store = Window.partitionBy("store_id")
q4_df = emp_sales.withColumn("avg_store_emp_sales", avg("emp_total_sales").over(w_store)) \
    .filter(col("emp_total_sales") > col("avg_store_emp_sales"))

q4_df.show(5)

+--------+-----------+------------------+-------------------+
|store_id|employee_id|   emp_total_sales|avg_store_emp_sales|
+--------+-----------+------------------+-------------------+
|      93|          3|             640.3|  616.6366666666667|
|      93|          2|1185.6299999999999|  616.6366666666667|
|       1|          1|             689.2|            387.555|
|      40|         60|            324.96|             241.94|
|      17|         96|1039.1999999999998|  438.0633333333333|
+--------+-----------+------------------+-------------------+
only showing top 5 rows



In [17]:
# Q5: Above-average revenue products with stock < 20
prod_rev = order_items.groupBy("product_id") \
    .agg(_sum(col("quantity") * col("price_per_unit")).alias("prod_revenue"))
avg_prod_rev = prod_rev.select(avg("prod_revenue")).collect()[0][0]

q5_df = products.join(prod_rev, "product_id") \
    .filter((col("prod_revenue") > avg_prod_rev) & (col("stock_quantity") < 20))

q5_df.show(5)

+----------+-------------+-----------+-----------+----------+----------+------------+--------------+------------+
|product_id| product_name|   category|subcategory|unit_price|cost_price|    supplier|stock_quantity|prod_revenue|
+----------+-------------+-----------+-----------+----------+----------+------------+--------------+------------+
|        88|VisionMax #88|Electronics|Televisions|   1405.15|     596.0|CircuitWorks|            16|     14051.5|
+----------+-------------+-----------+-----------+----------+----------+------------+--------------+------------+



In [18]:
# Q6: Highest revenue month for each region
region_monthly = orders.join(stores, "store_id") \
    .join(order_items, "order_id") \
    .withColumn("month", date_format("order_date", "yyyy-MM")) \
    .groupBy("region", "month") \
    .agg(_sum(col("quantity") * col("price_per_unit")).alias("monthly_revenue"))

w_region = Window.partitionBy("region").orderBy(col("monthly_revenue").desc())
q6_df = region_monthly.withColumn("rn", row_number().over(w_region)).filter(col("rn") == 1)

q6_df.show(5)

+------+-------+------------------+---+
|region|  month|   monthly_revenue| rn|
+------+-------+------------------+---+
| South|2025-07|           1563.77|  1|
|  East|2023-01| 7952.360000000001|  1|
|  West|2025-10| 7464.110000000001|  1|
| North|2024-03|4024.6499999999996|  1|
+------+-------+------------------+---+



In [19]:
# Q7: Customers who purchased from at least 3 distinct categories
q7_df = orders.join(order_items, "order_id") \
    .join(products, "product_id") \
    .groupBy("customer_id") \
    .agg(countDistinct("category").alias("distinct_categories")) \
    .filter(col("distinct_categories") >= 3)

q7_df.show(5)

+-----------+-------------------+
|customer_id|distinct_categories|
+-----------+-------------------+
|         91|                  3|
|         72|                  3|
|         23|                  4|
|         69|                  3|
|         97|                  5|
+-----------+-------------------+
only showing top 5 rows



In [20]:
# Q8: Supplier metrics sorted by total profit
q8_df = order_items.join(products, "product_id") \
    .groupBy("supplier") \
    .agg(
        countDistinct("product_id").alias("num_products"),
        _sum("quantity").alias("total_units_sold"),
        round(_sum(col("quantity") * col("price_per_unit")), 2).alias("total_revenue"),
        round(_sum(col("quantity") * (col("price_per_unit") - col("cost_price"))), 2).alias("total_profit")
    ) \
    .orderBy(col("total_profit").desc())

q8_df.show(5)

+-----------------+------------+----------------+-------------+------------+
|         supplier|num_products|total_units_sold|total_revenue|total_profit|
+-----------------+------------+----------------+-------------+------------+
|     CircuitWorks|           3|              19|     18598.09|     9993.35|
|     FitLife Inc.|           7|              38|     12117.77|     4698.66|
|  TechSource Inc.|           2|              10|      6150.54|     2938.64|
|GreenThumb Supply|           5|              19|      6046.57|     2790.37|
|  NovaTech Supply|           2|              15|      6462.42|     2771.88|
+-----------------+------------+----------------+-------------+------------+
only showing top 5 rows



In [21]:
# Q9: Top 3 product categories by profit within each region
q9_df = spark.sql("""
    WITH category_profit AS (
        SELECT s.region, p.category,
               SUM(oi.quantity * (oi.price_per_unit - p.cost_price)) AS total_profit,
               DENSE_RANK() OVER (PARTITION BY s.region ORDER BY SUM(oi.quantity * (oi.price_per_unit - p.cost_price)) DESC) as rnk
        FROM orders o
        JOIN stores s ON o.store_id = s.store_id
        JOIN order_items oi ON o.order_id = oi.order_id
        JOIN products p ON oi.product_id = p.product_id
        GROUP BY s.region, p.category
    )
    SELECT region, category, total_profit FROM category_profit WHERE rnk <= 3
""")

q9_df.show(5)

+------+-----------------+------------------+
|region|         category|      total_profit|
+------+-----------------+------------------+
| South|    Home & Garden|           1037.62|
| South|         Clothing|            281.43|
| South|Sports & Outdoors|            222.39|
|  East|      Electronics|12064.389999999998|
|  East|Sports & Outdoors|           2869.99|
+------+-----------------+------------------+
only showing top 5 rows



In [22]:
# Q10: Customers spending > average spending of their loyalty tier
q10_df = spark.sql("""
    WITH cust_spend AS (
        SELECT c.customer_id, c.loyalty_tier, SUM(oi.quantity * oi.price_per_unit) AS total_spent
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY c.customer_id, c.loyalty_tier
    )
    SELECT customer_id, loyalty_tier, total_spent
    FROM (
        SELECT *, AVG(total_spent) OVER (PARTITION BY loyalty_tier) as avg_tier_spend
        FROM cust_spend
    )
    WHERE total_spent > avg_tier_spend
""")

q10_df.show(5)

+-----------+------------+-----------+
|customer_id|loyalty_tier|total_spent|
+-----------+------------+-----------+
|         84|    Platinum|     599.08|
|         67|      Silver|    1547.22|
|         24|      Silver|    2928.61|
|         36|      Silver|    2169.93|
|         47|      Silver|    1216.81|
+-----------+------------+-----------+
only showing top 5 rows



In [23]:
# Q11: Products never ordered
q11_df = spark.sql("""
    SELECT p.* FROM products p
    LEFT JOIN order_items oi ON p.product_id = oi.product_id
    WHERE oi.product_id IS NULL
""")

q11_df.show(5)

+----------+-----------------+-------------+--------------+----------+----------+--------------+--------------+
|product_id|     product_name|     category|   subcategory|unit_price|cost_price|      supplier|stock_quantity|
+----------+-----------------+-------------+--------------+----------+----------+--------------+--------------+
|         1| Water Blaster #1| Toys & Games|  Outdoor Toys|     93.72|      61.6|KidsJoy Supply|            68|
|         3|    Floor Lamp #3|Home & Garden|      Lighting|    651.08|    274.81|     DecorPlus|            18|
|         6|Fantasy Figure #6| Toys & Games|Action Figures|     46.05|     31.01|KidsJoy Supply|           139|
|         7|      AudioMax #7|  Electronics|    Headphones|   1339.76|    884.45|  CircuitWorks|           173|
|         9|       FlexPad #9|  Electronics|       Tablets|    164.44|    100.45| GlobalElectro|           279|
+----------+-----------------+-------------+--------------+----------+----------+--------------+--------

In [24]:
# Q12: Employees who served customers from > 3 different cities
q12_df = spark.sql("""
    SELECT o.employee_id, COUNT(DISTINCT c.city) AS distinct_cities
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.employee_id
    HAVING COUNT(DISTINCT c.city) > 3
""")

q12_df.show(5)

+-----------+---------------+
|employee_id|distinct_cities|
+-----------+---------------+
|         57|              4|
|         59|              4|
|         69|              4|
+-----------+---------------+



In [25]:
# Q13: Highest revenue month for every store
q13_df = spark.sql("""
    WITH store_monthly AS (
        SELECT store_id, DATE_FORMAT(order_date, 'yyyy-MM') AS month,
               SUM(quantity * price_per_unit) AS revenue,
               ROW_NUMBER() OVER (PARTITION BY store_id ORDER BY SUM(quantity * price_per_unit) DESC) as rn
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY store_id, DATE_FORMAT(order_date, 'yyyy-MM')
    )
    SELECT store_id, month, revenue FROM store_monthly WHERE rn = 1
""")

q13_df.show(5)

+--------+-------+-------+
|store_id|  month|revenue|
+--------+-------+-------+
|      31|2025-10| 392.36|
|      85|2023-06|1216.81|
|      78|2023-10| 634.13|
|      34|2023-10| 852.82|
|      28|2023-08|  46.78|
+--------+-------+-------+
only showing top 5 rows



In [26]:
# Q14: Percentage contribution of every store to total revenue
q14_df = spark.sql("""
    WITH store_rev AS (
        SELECT store_id, SUM(quantity * price_per_unit) AS store_revenue
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY store_id
    )
    SELECT store_id, store_revenue,
           ROUND((store_revenue / SUM(store_revenue) OVER ()) * 100, 2) AS pct_contribution
    FROM store_rev
""")

q14_df.show(5)

2026-08-16 07:57:50,302 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-------------+----------------+
|store_id|store_revenue|pct_contribution|
+--------+-------------+----------------+
|      31|       392.36|            0.54|
|      85|      1216.81|            1.67|
|      78|       634.13|            0.87|
|      34|       852.82|            1.17|
|      28|        46.78|            0.06|
+--------+-------------+----------------+
only showing top 5 rows



In [27]:
# Q15: Customers with > 1 order on the same day
q15_df = spark.sql("""
    SELECT customer_id, CAST(order_date AS DATE) as order_day, COUNT(order_id) as orders_count
    FROM orders
    GROUP BY customer_id, CAST(order_date AS DATE)
    HAVING COUNT(order_id) > 1
""")

q15_df.show(5)

+-----------+---------+------------+
|customer_id|order_day|orders_count|
+-----------+---------+------------+
+-----------+---------+------------+



In [28]:
# Q16: Stores whose monthly revenue decreased compared to previous month
q16_df = spark.sql("""
    WITH store_trend AS (
        SELECT store_id, DATE_FORMAT(order_date, 'yyyy-MM') AS month,
               SUM(quantity * price_per_unit) AS current_month_rev,
               LAG(SUM(quantity * price_per_unit)) OVER (PARTITION BY store_id ORDER BY DATE_FORMAT(order_date, 'yyyy-MM')) as prev_month_rev
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY store_id, DATE_FORMAT(order_date, 'yyyy-MM')
    )
    SELECT store_id, month, current_month_rev, prev_month_rev
    FROM store_trend
    WHERE current_month_rev < prev_month_rev
""")

q16_df.show(5)

+--------+-------+------------------+------------------+
|store_id|  month| current_month_rev|    prev_month_rev|
+--------+-------+------------------+------------------+
|      93|2024-03|             23.98|1185.6299999999999|
|      17|2025-05|             10.39|1039.1999999999998|
|      55|2024-10|             10.08|            234.34|
|      21|2025-08|109.71000000000001|            3757.8|
|      32|2025-09|           1547.22| 7952.360000000001|
+--------+-------+------------------+------------------+
only showing top 5 rows

